# 모터-감속기 이상 탐지 — 학습 파이프라인 (Stage 2~4)

> **본격 학습 파이프라인의 학습부 단독 노트북** · Google Colab 용

Stage 1 에서 만든 **4피처 시계열 CSV** 를 입력으로, **(차종 × 센서) 슬롯별 모델**을 학습합니다.

`슬롯 라우팅 → chunker(2000) → StratifiedGroupKFold → MiniRocket(4채널) → LogisticRegression → ModelRegistry`

**사용법** — ① *환경 설정* 셀에서 sktime 설치 + Drive 마운트 → ② *경로 & 파라미터* 셀에서 `CSV_ROOT` / `MODEL_DIR` 두 곳만 채우기 → ③ 위에서 아래로 실행.

산출물: `model_registry.joblib`(슬롯별 모델), `eval_results.json`(평가).

## 0. 환경 설정 (Colab)

In [ ]:
# ── 환경 설정 (Colab) ────────────────────────────────────────────────
# MiniRocket(다변량)은 sktime 에 포함되어 있습니다.
!pip -q install -U sktime
# ※ 설치 후 import 에러가 나면: 런타임 다시 시작 → 이 셀부터 다시 실행

from google.colab import drive
drive.mount('/content/drive')

## 1. 경로 & 파라미터 설정

**`CSV_ROOT`(Stage 1 출력) 와 `MODEL_DIR`(저장 위치) 두 값만** 채우면 됩니다. 불균형 처리는 기본적으로 꺼져 있으며, `LR_CLASS_WEIGHT='balanced'` 또는 `DOWNSAMPLE=True` 로 켤 수 있습니다.

In [ ]:
# =====================================================================
#  경로 & 파라미터 설정
#  ── Stage 1(PNG→CSV) 결과를 입력으로 사용합니다 ──
# =====================================================================

# [필수] Stage 1 에서 생성한 CSV 들의 루트 (예: "/content/drive/MyDrive/aihub/motor_csv")
CSV_ROOT   = ""      # ← 비워둠. 채우기

# [필수] 학습된 모델 / 평가 결과 저장 경로 (예: "/content/drive/MyDrive/aihub/motor_models")
MODEL_DIR  = ""      # ← 비워둠. 채우기

# ---------------------------------------------------------------------
#  파라미터 (기본값 그대로 사용 가능)
# ---------------------------------------------------------------------
CHUNK_SIZE      = 2000          # chunker: 청크 1개 = 2000 행 (고정 길이 시계열 샘플)
FEATURE_COLS    = ["peak_freq", "band_start", "band_end", "rms"]  # 4채널 (multivariate)
CLASS_ORDER     = ["DEMAG", "ECC10", "ECC20", "NORMAL", "REDUC"]  # 5클래스 고정 순서
N_SPLITS        = 3             # StratifiedGroupKFold 분할 수 (≥3 권장)
NUM_KERNELS     = 10000         # MiniRocket 커널 수
RANDOM_STATE    = 42

# 불균형 처리 옵션 (둘 다 끄면 아무 처리 안 함 — 발표: "불균형 심할 시 다운샘플링")
LR_CLASS_WEIGHT = None          # "balanced" 로 두면 LogisticRegression 클래스 가중치 적용
DOWNSAMPLE      = False         # True 면 다수 클래스 청크를 다운샘플링
DOWNSAMPLE_TO   = "min"         # "min"(최소 클래스 수) | "median"

## 2. 데이터 유틸 — 스캔 · 슬롯 분류 · 청킹

CSV 를 `(vehicle, sensor)` 슬롯으로 묶고, 각 CSV(연속 세션)를 `CHUNK_SIZE`(2000) 행씩 잘라 고정 길이 다변량 샘플로 만듭니다. 각 청크는 원본 CSV 의 `group_key` 를 상속해 **데이터 누수 방지**에 사용됩니다.

In [ ]:
# =====================================================================
#  데이터 유틸 — CSV 스캔 · 슬롯 분류 · 청킹
#   슬롯 = (vehicle, sensor)  →  최대 3차종 × 3센서 = 9개
#   청크 = 한 CSV(연속 세션)를 CHUNK_SIZE 행씩 자른 고정 길이 샘플
# =====================================================================
import os, glob
import numpy as np
import pandas as pd
from collections import defaultdict
from pathlib import Path


def find_csv_paths(root):
    return sorted(glob.glob(os.path.join(root, "**", "*.csv"), recursive=True))


def slot_of_csv(path):
    """CSV 첫 행에서 (vehicle, sensor, fault_class) 읽기 (가벼움)."""
    head = pd.read_csv(path, nrows=1, usecols=["vehicle", "sensor", "fault_class"])
    return (str(head["vehicle"].iloc[0]),
            str(head["sensor"].iloc[0]),
            str(head["fault_class"].iloc[0]))


def scan_slots(root):
    """모든 CSV 를 (vehicle, sensor) 슬롯으로 묶고, 슬롯별 등장 클래스 집합을 기록."""
    slots, slot_classes = defaultdict(list), defaultdict(set)
    for p in find_csv_paths(root):
        try:
            v, s, fc = slot_of_csv(p)
        except Exception as e:
            print("스킵(헤더 읽기 실패):", p, "→", e)
            continue
        slots[(v, s)].append(p)
        slot_classes[(v, s)].add(fc)
    return slots, slot_classes


def load_slot_chunks(paths):
    """
    슬롯에 속한 CSV 들을 읽어 청크 단위 샘플로 변환.
    반환: X (n_chunks, C, CHUNK_SIZE), y (n_chunks,), groups (n_chunks,) | None
    각 청크의 label = 해당 CSV 의 fault_class, group = group_key (누수 방지용).
    """
    Xs, ys, gs = [], [], []
    for p in paths:
        try:
            df = pd.read_csv(p)
        except Exception as e:
            print("스킵(읽기 실패):", p, "→", e); continue
        if df.empty:
            continue
        feats = df[FEATURE_COLS].to_numpy(dtype=np.float32)     # (n, C)
        n = (len(feats) // CHUNK_SIZE) * CHUNK_SIZE             # 끝의 잔여행(<2000) 버림
        if n == 0:
            continue
        C = feats.shape[1]
        chunks = feats[:n].T.reshape(C, -1, CHUNK_SIZE).transpose(1, 0, 2)  # (n_chunks, C, L)
        label = str(df["fault_class"].iloc[0])
        group = str(df["group_key"].iloc[0])
        Xs.append(chunks)
        ys += [label] * len(chunks)
        gs += [group] * len(chunks)
    if not Xs:
        return None
    return np.concatenate(Xs, 0), np.array(ys), np.array(gs)

## 3. 슬롯 스캔 — 학습 가능 여부 확인

본 학습 전에 슬롯별 CSV 수와 등장 클래스를 출력합니다. **클래스가 1개뿐인 슬롯은 분류가 불가능**하므로 학습에서 자동 제외됩니다.

In [ ]:
# =====================================================================
#  슬롯 스캔 실행 — 어떤 슬롯이 학습 가능한지 먼저 확인
# =====================================================================
if not CSV_ROOT:
    raise ValueError("CSV_ROOT(Stage 1 출력 CSV 경로)를 먼저 채워주세요.")

slots, slot_classes = scan_slots(CSV_ROOT)
print(f"발견된 슬롯(차종 × 센서): {len(slots)}개\n")
for k in sorted(slots, key=lambda x: (str(x[0]), str(x[1]))):
    cls = sorted(slot_classes[k])
    flag = "" if len(cls) >= 2 else "   ⚠ 단일 클래스 → 분류 학습 불가(스킵 예정)"
    print(f"  {str(k[0]):6s} · {str(k[1]):10s} | CSV {len(slots[k]):4d}개 | 클래스 {cls}{flag}")

## 4. 모델 정의 — MiniRocket(다변량) + LogisticRegression

채널별 표준화(스케일 차 보정) → MiniRocket 특징 변환 → StandardScaler → LogisticRegression. 예측은 항상 **5클래스(CLASS_ORDER) 정렬 확률**을 반환합니다 (슬롯에 없던 클래스는 0). `StratifiedGroupKFold` 로 group 누수 방지 + 클래스 비율 유지를 동시에 달성하고, 마지막에 전체 데이터로 재학습한 모델을 저장합니다.

In [ ]:
# =====================================================================
#  모델 정의 — MiniRocket(다변량) + LogisticRegression
#   · 채널별 표준화(스케일 차이 보정) → MiniRocket → StandardScaler → LR
#   · predict 는 항상 5클래스(CLASS_ORDER) 정렬 확률을 반환 (없는 클래스는 0)
# =====================================================================
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


def make_minirocket(n_channels):
    if n_channels == 1:
        from sktime.transformations.panel.rocket import MiniRocket
        return MiniRocket(num_kernels=NUM_KERNELS, random_state=RANDOM_STATE)
    from sktime.transformations.panel.rocket import MiniRocketMultivariate
    return MiniRocketMultivariate(num_kernels=NUM_KERNELS, random_state=RANDOM_STATE)


def _fit_channel_stats(X):
    mean = X.mean(axis=(0, 2)).astype(np.float32)
    std  = X.std(axis=(0, 2)).astype(np.float32)
    std[std == 0] = 1.0
    return mean, std


def _apply_channel(X, mean, std):
    return ((X - mean[None, :, None]) / std[None, :, None]).astype(np.float32)


def fit_components(Xtr, ytr):
    """슬롯 하나의 학습 구성요소를 적합. 반환 dict 가 곧 '저장될 모델'."""
    mean, std = _fit_channel_stats(Xtr)
    Xn = _apply_channel(Xtr, mean, std)
    mr = make_minirocket(Xn.shape[1]); mr.fit(Xn)
    Ftr = np.asarray(mr.transform(Xn))
    sc = StandardScaler().fit(Ftr)
    lr = LogisticRegression(max_iter=2000, class_weight=LR_CLASS_WEIGHT)
    lr.fit(sc.transform(Ftr), ytr)
    return dict(ch_mean=mean, ch_std=std, mr=mr, sc=sc, lr=lr,
                classes=list(map(str, lr.classes_)))


def slot_proba(model, X):
    """(n, 5) — CLASS_ORDER 순서로 정렬된 확률 (학습에 없던 클래스는 0)."""
    Xn = _apply_channel(X, model["ch_mean"], model["ch_std"])
    F = model["sc"].transform(np.asarray(model["mr"].transform(Xn)))
    p = model["lr"].predict_proba(F)
    full = np.zeros((len(X), len(CLASS_ORDER)), dtype=float)
    for j, c in enumerate(model["lr"].classes_):
        full[:, CLASS_ORDER.index(str(c))] = p[:, j]
    return full


def slot_predict(model, X):
    return np.array(CLASS_ORDER)[slot_proba(model, X).argmax(1)]


def maybe_downsample(X, y, g):
    if not DOWNSAMPLE:
        return X, y, g
    rng = np.random.default_rng(RANDOM_STATE)
    classes, counts = np.unique(y, return_counts=True)
    target = counts.min() if DOWNSAMPLE_TO == "min" else int(np.median(counts))
    keep = []
    for c in classes:
        idx = np.where(y == c)[0]
        if len(idx) > target:
            idx = rng.choice(idx, target, replace=False)
        keep.append(idx)
    keep = np.sort(np.concatenate(keep))
    return X[keep], y[keep], g[keep]


def train_slot(slot_key, X, y, g):
    """슬롯 1개 학습: StratifiedGroupKFold 교차검증 + 전체 데이터 재학습(최종 모델)."""
    from sklearn.model_selection import StratifiedGroupKFold
    from sklearn.metrics import f1_score, accuracy_score, confusion_matrix

    info = dict(slot=slot_key, n_chunks=int(len(y)), classes=sorted(set(map(str, y))))
    if len(set(y)) < 2:
        info["status"] = "skipped: 단일 클래스 (분류 불가)"
        return None, info

    X, y, g = maybe_downsample(X, y, g)
    labels = sorted(set(map(str, y)))

    # 클래스별 그룹 수로 가능한 분할 수 결정 (StratifiedGroupKFold 제약)
    min_groups = min(len(set(g[y == c])) for c in labels)
    n_splits = min(N_SPLITS, min_groups)

    if n_splits >= 2:
        sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
        f1s, accs, cm_total = [], [], None
        for tr, va in sgkf.split(np.zeros(len(y)), y, g):
            comp = fit_components(X[tr], y[tr])
            pred = slot_predict(comp, X[va])
            f1s.append(f1_score(y[va], pred, average="macro", labels=labels, zero_division=0))
            accs.append(accuracy_score(y[va], pred))
            cm = confusion_matrix(y[va], pred, labels=labels)
            cm_total = cm if cm_total is None else cm_total + cm
        info.update(status="ok", n_splits=int(n_splits), labels=labels,
                    macro_f1_mean=float(np.mean(f1s)), macro_f1_std=float(np.std(f1s)),
                    acc_mean=float(np.mean(accs)), acc_std=float(np.std(accs)),
                    confusion_matrix=cm_total.tolist())
    else:
        info["status"] = f"trained (CV 생략: 클래스당 그룹 수 부족 min_groups={min_groups})"
        info["labels"] = labels

    final_model = fit_components(X, y)          # 배포용: 전체 데이터로 재학습
    return final_model, info

## 5. 전체 학습 실행 → ModelRegistry 저장

모든 슬롯을 학습해 `{(vehicle, sensor): model}` 형태의 ModelRegistry 로 저장합니다. 학습 불가 슬롯은 사유와 함께 평가 결과에 기록됩니다.

In [ ]:
# =====================================================================
#  전체 학습 실행 → ModelRegistry + 평가 결과 저장
# =====================================================================
import json, joblib
from tqdm.auto import tqdm

if not MODEL_DIR:
    raise ValueError("MODEL_DIR(저장 경로)를 먼저 채워주세요.")
os.makedirs(MODEL_DIR, exist_ok=True)

registry, eval_rows = {}, []
for slot_key in tqdm(sorted(slots, key=lambda x: (str(x[0]), str(x[1]))), desc="슬롯 학습"):
    data = load_slot_chunks(slots[slot_key])
    if data is None:
        eval_rows.append(dict(slot=slot_key, n_chunks=0, status="skipped: 청크 없음"))
        continue
    X, y, g = data
    model, info = train_slot(slot_key, X, y, g)
    eval_rows.append(info)
    if model is not None:
        registry[slot_key] = model

# ModelRegistry: {(vehicle, sensor): model_dict}
joblib.dump(registry, os.path.join(MODEL_DIR, "model_registry.joblib"))
with open(os.path.join(MODEL_DIR, "eval_results.json"), "w", encoding="utf-8") as f:
    json.dump(eval_rows, f, ensure_ascii=False, indent=2, default=str)

print(f"\n학습 완료 — 등록된 모델 {len(registry)}개")
print("저장:", os.path.join(MODEL_DIR, "model_registry.joblib"))

## 6. 평가 요약

슬롯별 macro-F1(폴드 평균 ± 표준편차), accuracy, per-class F1, pooled confusion matrix 를 출력하고 히트맵으로 시각화합니다.

In [ ]:
# =====================================================================
#  평가 요약 — macro-F1 / per-class F1 / confusion matrix / fold 분산
# =====================================================================
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

trained = [r for r in eval_rows if "macro_f1_mean" in r]

print("=== 학습된 슬롯 요약 ===")
if trained:
    summ = pd.DataFrame([{
        "slot":     f'{r["slot"][0]}·{r["slot"][1]}',
        "n_chunks": r["n_chunks"],
        "classes":  len(r["labels"]),
        "n_splits": r["n_splits"],
        "macro_F1": f'{r["macro_f1_mean"]:.3f} ± {r["macro_f1_std"]:.3f}',
        "accuracy": f'{r["acc_mean"]:.3f} ± {r["acc_std"]:.3f}',
    } for r in trained])
    print(summ.to_string(index=False))
else:
    print("CV 평가가 수행된 슬롯이 없습니다.")

for r in eval_rows:
    if "macro_f1_mean" not in r:
        print(f'[참고] {r["slot"]}: {r.get("status")}  (chunks={r.get("n_chunks")}, classes={r.get("classes")})')


def per_class_f1(cm, labels):
    cm = np.array(cm); out = {}
    for i, lab in enumerate(labels):
        tp = cm[i, i]; fp = cm[:, i].sum() - tp; fn = cm[i, :].sum() - tp
        p   = tp / (tp + fp) if (tp + fp) else 0.0
        rcl = tp / (tp + fn) if (tp + fn) else 0.0
        out[lab] = round(2 * p * rcl / (p + rcl), 3) if (p + rcl) else 0.0
    return out


for r in trained:
    cm, labels = np.array(r["confusion_matrix"]), r["labels"]
    print(f'\n■ {r["slot"][0]} · {r["slot"][1]}  (pooled CV confusion matrix)')
    print(pd.DataFrame(cm, index=[f"T:{l}" for l in labels],
                       columns=[f"P:{l}" for l in labels]).to_string())
    print("per-class F1:", per_class_f1(cm, labels))

# confusion matrix 히트맵 그리드
if trained:
    try:
        n = len(trained); cols = min(3, n); rows = (n + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3.4 * rows), squeeze=False)
        for ax in axes.ravel():
            ax.axis("off")
        for k, r in enumerate(trained):
            ax = axes.ravel()[k]; ax.axis("on")
            cm, labels = np.array(r["confusion_matrix"]), r["labels"]
            ax.imshow(cm, cmap="Blues")
            ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
            ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=8)
            ax.set_title(f'{r["slot"][0]}·{r["slot"][1]}', fontsize=9)
            for i in range(len(labels)):
                for j in range(len(labels)):
                    ax.text(j, i, int(cm[i, j]), ha="center", va="center", fontsize=7,
                            color="white" if cm[i, j] > cm.max() / 2 else "black")
        plt.tight_layout(); plt.show()
    except Exception as e:
        print("히트맵 생략:", e)

## 7. (옵션) 예측 데모 — Anomaly Score

저장된 레지스트리를 불러와 검증 청크 몇 개를 예측하고, `AS = 1 − P(NORMAL)` 와 5클래스 확률을 확인합니다.

In [ ]:
# =====================================================================
#  (옵션) 예측 데모 — 5클래스 확률 + Anomaly Score
#   AS = 1 − P(NORMAL)
# =====================================================================
import joblib, numpy as np

reg = joblib.load(os.path.join(MODEL_DIR, "model_registry.joblib"))
if not reg:
    print("등록된 모델이 없습니다.")
else:
    key = next(iter(reg))
    X, y, g = load_slot_chunks(slots[key])
    k = min(8, len(X))
    proba = slot_proba(reg[key], X[:k])
    ni = CLASS_ORDER.index("NORMAL")
    print(f"슬롯 {key} — 청크 {k}개 예측\n")
    for i in range(k):
        AS = 1 - proba[i, ni]
        top = CLASS_ORDER[proba[i].argmax()]
        probs = " ".join(f"{c}:{proba[i, j]:.2f}" for j, c in enumerate(CLASS_ORDER))
        print(f"  실제={y[i]:7s} 예측={top:7s} AS={AS:.3f} | {probs}")

    # 참고: 대시보드 통합점수 = 0.6 × (센서 AS 평균) + 0.4 × (센서 AS 최고)
    print("\n※ 통합점수(대시보드)는 한 차종의 3개 센서 AS를 0.6×평균 + 0.4×최고 로 합산합니다.")